# 01 — Cleaning

Turns `data/raw.parquet` (both sheets of UCI Online Retail II, built by `python src/load_data.py`)
into `data/clean.parquet` and `data/cancellations.parquet`.

The logic lives in `src/clean.py` so later notebooks and the churn features use exactly the same rules.
Steps run in this order, and each one logs how many rows it removed:

1. Drop rows with no `Customer ID`: the analysis is per customer, and these rows can't be attributed to anyone.
2. Set cancellations (`Invoice` starting with "C") aside. They are not sales, but they feed a per-customer return-rate feature later.
3. Drop `Quantity <= 0` and `Price <= 0`.
4. Drop exact duplicate rows.
5. Drop non-product `StockCode`s (postage, fees, manual adjustments, test items).
6. Add `Revenue = Quantity * Price` and cast `Customer ID` to int.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import clean as cl
from src.metrics import save_metrics

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)

In [2]:
raw, df, cancels, log_df, info = cl.run(save=True)
print(f"raw rows: {len(raw):,}   clean rows: {len(df):,}   cancellation rows: {len(cancels):,}")

raw rows: 1,067,371   clean rows: 776,579   cancellation rows: 18,744


## Cleaning log

In [3]:
total_removed = len(raw) - len(df)
summary = pd.DataFrame([{
    "step": "TOTAL", "rows_before": len(raw), "rows_after": len(df), "removed": total_removed,
    "pct_removed": round(100 * total_removed / len(raw), 2), "reason": "all steps",
}])
cleaning_log = pd.concat([log_df, summary], ignore_index=True)
cleaning_log

,step,rows_before,rows_after,removed,pct_removed,reason
0,drop_null_customer_id,1067371,824364,243007,22.77,revenue can't be attributed to a customer
1,separate_cancellations,824364,805620,18744,2.27,invoice starts with 'C'; saved to cancellations.parquet
2,drop_nonpositive_qty_price,805620,805549,71,0.01,Quantity <= 0 or Price <= 0 is not a sale
3,drop_exact_duplicates,805549,779425,26124,3.24,identical rows are double-logged lines
4,drop_non_product_codes,779425,776579,2846,0.37,"postage, fees, adjustments and test codes are not products"
5,TOTAL,1067371,776579,290792,27.24,all steps


## Cancellations

Rates are measured on customer-attributed rows (after step 1).
By revenue = value of cancelled lines ÷ gross sales value.

In [4]:
pd.Series(info["cancellation"])

cancelled_invoices                 7901.00
total_invoices                    44876.00
cancel_rate_by_invoices_pct          17.61
cancelled_revenue               1095136.79
gross_revenue                  17743429.18
cancel_rate_by_revenue_pct            6.17
dtype: float64

## Non-product stock codes

The drop list (`NON_PRODUCT_CODES` and prefixes `ADJUST`, `TEST`, `GIFT_` in `src/clean.py`) came from listing every
code that isn't the usual 5-digit product format and reading its description. `SP1002` (a kid's chalkboard) is
non-standard but is a real product, so it stays. These are the codes actually found and dropped at step 5.
Counts are after steps 1–4, so codes that mostly appear on cancellation invoices (e.g. `D`, discount) look small here.

In [5]:
pd.Series(info["non_product_codes_dropped"], name="rows_dropped").rename_axis("StockCode").to_frame()

,rows_dropped
StockCode,
POST,1803
M,681
C2,248
ADJUST,32
BANK CHARGES,31
PADS,17
DOT,16
TEST001,9
D,5


## Verification

In [6]:
pct_removed = 100 * total_removed / len(raw)
print(f"total removed: {total_removed:,} rows ({pct_removed:.1f}%)")
# Expect roughly 20-30%; almost all of it comes from missing Customer IDs
assert 15 <= pct_removed <= 35, pct_removed

key_cols = ["Invoice", "StockCode", "Quantity", "InvoiceDate", "Price", "Customer ID", "Country", "Revenue"]
nulls = df[key_cols].isna().sum()
print("nulls in key columns:", nulls.sum())
assert nulls.sum() == 0

assert (df["Quantity"] > 0).all() and (df["Price"] > 0).all()
assert not df["Invoice"].str.startswith("C").any()
assert df["Customer ID"].dtype.kind == "i"

print("raw date range:  ", raw["InvoiceDate"].min(), "->", raw["InvoiceDate"].max())
print("clean date range:", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())
assert df["InvoiceDate"].min().date() == raw["InvoiceDate"].min().date()
assert df["InvoiceDate"].max().date() == raw["InvoiceDate"].max().date()

total removed: 290,792 rows (27.2%)
nulls in key columns: 0
raw date range:   2009-12-01 07:45:00 -> 2011-12-09 12:50:00
clean date range: 2009-12-01 07:45:00 -> 2011-12-09 12:50:00


In [7]:
print(f"customers: {df['Customer ID'].nunique():,}")
print(f"invoices:  {df['Invoice'].nunique():,}")
print(f"products:  {df['StockCode'].nunique():,}")
print(f"countries: {df['Country'].nunique():,}")
print(f"revenue:   £{df['Revenue'].sum():,.0f}")
df.head()

customers: 5,852
invoices:  36,594
products:  4,620
countries: 41
revenue:   £17,068,583


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


## Record key numbers for RESULTS.md

In [8]:
save_metrics("01_clean", {
    "raw_rows": len(raw),
    "clean_rows": len(df),
    "rows_removed": total_removed,
    "pct_removed": round(pct_removed, 2),
    "cleaning_log": log_df.to_dict(orient="records"),
    "cancellation": info["cancellation"],
    "non_product_codes_dropped": {k: int(v) for k, v in info["non_product_codes_dropped"].items()},
    "customers": int(df["Customer ID"].nunique()),
    "invoices": int(df["Invoice"].nunique()),
    "products": int(df["StockCode"].nunique()),
    "countries": int(df["Country"].nunique()),
    "revenue_gbp": round(float(df["Revenue"].sum()), 2),
    "date_min": str(df["InvoiceDate"].min()),
    "date_max": str(df["InvoiceDate"].max()),
})

saved metrics/01_clean.json (14 keys)
